In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
import re
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
#Loading and preparing injury DF
injury_df = pd.read_csv("Injury2021-2024.csv")


Injury Dataset Transformations
The injury dataset contains the injury/status report of players before every game. In order to identify each injury and not over count, I will identify when a new injury begins and keep track of how many games they miss

Two seperate datasets are being created for a clustering model (unsupervised learning) classifcation model (supervised) and Data visualizations

General Transformations
- Normalize player names
- Create season variable
- Drop non-injury rows
- Creating injury location and type variable
- Season phase variable for time of injury
- Count of prior injruies per player/season

Dataset A (Injury Event, ML models)
- Collapsed down to each injury event

Dataset B (Raw Injury Report, For Visualziations)


In [ ]:
#Injury Dataset Transformation
#Standardizing Name
def fix_name(name):
    if pd.isna(name):
        return name
    if "," in name:
        last, first = name.split(",", 1)
        return first.strip() + " " + last.strip()
    else:
        return name.strip()
    
injury_df['PLAYER'] = injury_df['PLAYER'].apply(fix_name)

def normalize_initials(name):
    if pd.isna(name):
        return name
    
    # Normalize common initial patterns BEFORE fix_name()
    # Examples:
    # A.J. Green → AJ Green
    # A. J. Green → AJ Green
    # A.J Green → AJ Green
    name = re.sub(r"\b([A-Za-z])\.\s*([A-Za-z])\.\b", r"\1\2", name)  # A.J. -> AJ
    name = re.sub(r"\b([A-Za-z])\.\b", r"\1", name)                  # A. -> A
    name = name.replace(".", "")                                     # Remove remaining periods
    
    return name.strip()

injury_df["PLAYER"] = injury_df["PLAYER"].apply(normalize_initials)



#Creating Season Variable
injury_df['DATE'] = pd.to_datetime(injury_df['DATE'])
injury_df['Year'] = injury_df['DATE'].dt.year
injury_df['Month'] = injury_df['DATE'].dt.month
injury_df['Day'] = injury_df['DATE'].dt.day

#October of year 0 to June of year 1 is full season
#Season 2022-2024
#If month is Oct, Nov, Dec, then the season is the following year, else it is the current year
injury_df['Season'] = np.where(injury_df['Month'] >= 10, injury_df['Year'] + 1, injury_df['Year'])

#Dropping non-injury Rows and keeping only injuries that resulted in player being "Out"
injury_df = injury_df[injury_df['REASON'].str.contains("Injury/Illness", case = False, na = False)]
injury_df = injury_df[injury_df['STATUS'].str.contains("Out", case = False, na = False)]



#Creating body part and injury detail variable
# Body Part
injury_df['BodyPart'] = injury_df['REASON'].str.extract(r'-\s*(.*?);', expand=False)


#Creating grouping for body part, its really messy
injury_df["BodyPart_clean"] = (
    injury_df["BodyPart"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9/\-\s]", "", regex=True)     # remove weird quotes / unicode
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

def map_body_part_group(x):
    if pd.isna(x) or x in ["", "na", "n/a", "not available", '""', '" "', '-']:
        return "Other"
    
    # LOWER BODY
    if any(k in x for k in [
        "ankle","foot","toe","knee","hamstring","quad","thigh","calf","shin",
        "achilles","hip","groin","adductor","tibia","fibula","leg","midfoot"
    ]):
        return "Lower Body"

    # UPPER BODY
    if any(k in x for k in [
        "shoulder","elbow","wrist","hand","finger","thumb","forearm",
        "arm","bicep","tricep","metacarpal","radius","ulna"
    ]):
        return "Upper Body"

    # SPINE
    if any(k in x for k in [
        "back","lumbar","spine","cervical","disc","l5","l4","mid back","low back"
    ]):
        return "Spine"

    # CORE
    if any(k in x for k in [
        "abdomen","abdominal","core","oblique","rib","stomach"
    ]):
        return "Core"

    # HEAD / FACE
    if any(k in x for k in [
        "concussion","head","face","facial","eye","jaw","nose","orbital","dental"
    ]):
        return "Head/Face"

    # RESPIRATORY / CHEST
    if any(k in x for k in [
        "chest","sternum","lung","pneumothorax"
    ]):
        return "Chest/Lung"

    # ILLNESS / SYSTEMIC
    if any(k in x for k in [
        "illness","covid","flu","overall body","general"
    ]):
        return "Illness"

    return "Other"


injury_df["BodyPartGroup"] = injury_df["BodyPart_clean"].apply(map_body_part_group)



# Injury Detail
injury_df['InjuryDetail'] = injury_df['REASON'].str.extract(r';\s*(.*)', expand=False)

#Season Phase variable
def season_phase(m):
    if m in [10, 11]:
        return "Early"
    elif m in [12, 1]:
        return "Mid"
    else:
        return "Late"
injury_df['SeasonPhase'] = injury_df['Month'].apply(season_phase)

#Count of injuries by player/season
injury_df = injury_df.sort_values(['PLAYER', 'Season', 'DATE'])

injury_df['InjuriesThisSeason'] = (
    injury_df.groupby(['PLAYER', 'Season']).cumcount()
)



injury_df.to_csv("visualization_injuries_2021_2024.csv", index=False)

For the ML Model dataset, I need to collapse down to injury event. I consider an injury apart of the same event if SAME player, SAME BodyPart, Injury rows reasonably close in time, No prolonged return to play period in between them (>10 days)

In [ ]:
#ML Model Dataset
event_df = injury_df.copy()

print(event_df.columns)
#Sort
event_df = event_df.sort_values(["PLAYER", "BodyPart", "InjuryDetail", "DATE"])

# Difference between consecutive injury reports
event_df["date_diff"] = event_df.groupby(
    ["PLAYER", "BodyPart"]
)["DATE"].diff().dt.days

#new injury if gap is > 10 days or first occurence
event_df["new_injury_flag"] = event_df["date_diff"].isna() | (event_df["date_diff"] > 10)

#create an injury event id to track every unique case
event_df["injury_event_id"] = event_df.groupby(
    ["PLAYER", "BodyPart"]
)["new_injury_flag"].cumsum()

#Collapsing the data down to each incident
collapsed = event_df.groupby(
    ["PLAYER", "Season", "BodyPart", "BodyPart_clean", "BodyPartGroup", "injury_event_id"]
).agg({
    "DATE": ["min", "max", "count"],
    "SeasonPhase": "first",
    "InjuryDetail": "first",
    "REASON": "first",
    "TEAM": "first",
    "GAME": "first",
    "STATUS": "first"
})


collapsed.columns = [
    "InjuryStart", "InjuryEnd", "GamesMissed",
    "SeasonPhase", "InjuryDetail", "Reason",
    "Team", "Game", "Status"
]
collapsed = collapsed.reset_index()

#Duration of Injury
collapsed["DurationDays"] = (collapsed["InjuryEnd"] - collapsed["InjuryStart"]).dt.days


# Event-based injury count
collapsed = collapsed.sort_values(["PLAYER", "Season", "InjuryStart"])
collapsed["InjuriesThisSeason"] = collapsed.groupby(
    ["PLAYER", "Season"]
).cumcount()

collapsed.to_csv("ml_injury_events_2021_2024.csv", index=False)

Merging in the season stats data

In [ ]:
s22 = pd.read_csv("2021_Stats.csv")
s23 = pd.read_csv("2022_Stats.csv")
s24 = pd.read_csv("2023_Stats.csv")

In [ ]:
#assigning season 
s22["Season"] = 2022
s23["Season"] = 2023
s24["Season"] = 2024
stats = pd.concat([s22, s23, s24], ignore_index=True)


#Flag players that were traded
stats["has_multi_team_total"] = stats["Team"].str.contains("TM", case=False, na=False)

final_rows = []


for (player, season), group in stats.groupby(["Player", "Season"]):
    # Case 1 — traded player, so keep the total row only
    if group["has_multi_team_total"].any():
        row = group[group["has_multi_team_total"]].iloc[0]
        final_rows.append(row)
    
    # Case 2 — single-team season, so keep first row
    else:
        row = group.iloc[0]
        final_rows.append(row)

stats_clean = pd.DataFrame(final_rows)


# Create player_key to match injury dataset
stats_clean["player_clean"] = stats_clean["Player"].str.strip()
stats_clean["player_key"] = (
    stats_clean["player_clean"]
    .str.lower()
    .str.replace("[^a-z ]","", regex=True)
    .str.replace("\s+"," ", regex=True)
    .str.strip()
)

#Create Avg Mins Per Game Varaible
stats_clean["MPG"] = stats_clean["MP"]/stats_clean["G"]

#Create Avg Points per Game Variable
stats_clean["AVGPTS"] = stats_clean["PTS"]/stats_clean["G"]

#Create Avg FTS per Game Variable
stats_clean["AVGFTS"] = stats_clean["FT"]/stats_clean["G"]

#Create Avg 3PTS per Game Variable
stats_clean["AVG3PTS"] = stats_clean["3P"]/stats_clean["G"]




stats_clean.to_csv("stats_2021_2024.csv", index=False)


In [ ]:
# Load datasets
ml_df = pd.read_csv("ml_injury_events_2021_2024.csv")
viz_df = pd.read_csv("visualization_injuries_2021_2024.csv")
stats_clean = pd.read_csv("stats_2021_2024.csv")

#Normalizing both names, have to remove speical characters
def normalize_for_merge(name):
    if pd.isna(name):
        return name

    # lowercase + strip
    name = name.lower().strip()

    # remove accents (Dončić to doncic)
    name = ''.join(
        c for c in unicodedata.normalize('NFKD', name)
        if not unicodedata.combining(c)
    )

    # convert curly apostrophes ’ to '
    name = name.replace("’", "'")

    # remove periods entirely (A.J. to aj)
    name = name.replace(".", "")

    # keep letters, numbers, spaces, periods, hyphens, apostrophes
    name = re.sub(r"[^a-z0-9\.\-'\s]", "", name)

    # collapse multiple spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

ml_df["merge_key"] = ml_df["PLAYER"].apply(normalize_for_merge)
viz_df["merge_key"] = viz_df["PLAYER"].apply(normalize_for_merge)
stats_clean["merge_key"] = stats_clean["Player"].apply(normalize_for_merge)



# merging ml injury data with stats
merged_ml_df = ml_df.merge(
    stats_clean,
    left_on=["merge_key", "Season"],
    right_on=["merge_key", "Season"],
    how="left",
    indicator=True
)

# merging viz injury data with stats
merged_viz_df = viz_df.merge(
    stats_clean,
    left_on=["merge_key", "Season"],
    right_on=["merge_key", "Season"],
    how="left",
    indicator=True
)

# Extract missing rows
missing = merged_ml_df[merged_ml_df["_merge"] == "left_only"]

# Show first 20 unmatched player names to find possibnle issues
missing_players = missing["PLAYER"].unique()
missing_players[:20]

# Extract missing rows
missing = merged_viz_df[merged_viz_df["_merge"] == "left_only"]

# Show first 20 unmatched player names
missing_players = missing["PLAYER"].unique()
missing_players[:20]


#drop unmatched rows
ml_final = merged_ml_df[merged_ml_df["_merge"] == "both"].copy()
viz_final = merged_viz_df[merged_viz_df["_merge"] == "both"].copy()

#Create a lag variable for ml dataaset

#Here I am making a lag variable
history_df = ml_final[['PLAYER', 'Season', 'InjuriesThisSeason', 'Rk']].drop_duplicates()
history_df['Next_Season'] = history_df['Season'] + 1
history_df = history_df.rename(columns={
    'InjuriesThisSeason': 'Injuries_Prev_Season',
    'Rk': 'Rank_Prev_Season'
})
ml_final = ml_final.merge(
    history_df[['PLAYER', 'Next_Season', 'Injuries_Prev_Season', 'Rank_Prev_Season']],
    left_on=['PLAYER', 'Season'],
    right_on=['PLAYER', 'Next_Season'],
    how='left'
)

ml_final['Injuries_Prev_Season'] = ml_final['Injuries_Prev_Season'].fillna(0)
ml_final['Rank_Prev_Season'] = ml_final['Rank_Prev_Season'].fillna(-1)
ml_final.drop(columns=['Next_Season'], inplace=True)


In [ ]:
#Missing Value problem
print(len(ml_final))
ml_final.isna().sum().sort_values(ascending=False)
#Awards is main source of problem
ml_final = ml_final.drop(columns = ["Awards", "BodyPart_clean"])
ml_final = ml_final.dropna()
print(len(ml_final))


In [ ]:
ml_final.to_csv("final_ml_dataset_with_stats.csv", index=False)
viz_final.to_csv("final_viz_dataset_with_stats.csv", index=False)

Machine Learning
- Unsupervised Learning K- means clsutering, goal is to find player archtypes for injury

In [ ]:
# need to create a new data frame for this, where we take only the last record of every player each season
# goal is to not have a row for every injury event, but one row for each of these injured players during a season
df_profiles = ml_final.sort_values(['PLAYER', 'Season', 'InjuriesThisSeason']).drop_duplicates(subset=['PLAYER', 'Season'], keep='last').copy()

#select features for clustering
features = ['Age', 'MPG', 'InjuriesThisSeason', 'AVGPTS', 'AVGFTS', 'AVG3PTS', 'Pos']
#assign features and drop any nas that can cause errors
X_cluster = df_profiles[features].dropna()
num_features = ['Age', 'MPG', 'InjuriesThisSeason', 'AVGPTS', 'AVGFTS', 'AVG3PTS']
cat_features = ['Pos']

preprocessor_cluster = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

#Optimize the kmeans by finding optimal k through SSD

ssd_vec = []
cluster_range = range(1,20)
for k in cluster_range:
#kmeans model, finding the optimal k
    kmeans_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_cluster),
        ('cluster', KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    kmeans_pipeline.fit(X_cluster)

    ssd = kmeans_pipeline.named_steps['cluster'].inertia_
    ssd_vec.append(ssd)

# Plotting to find knee point
plt.figure(figsize=(10, 6))
plt.plot(cluster_range, ssd_vec, marker='o', linestyle='-', color='b')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Sum of Squared Distances (Inertia)')
plt.xticks(cluster_range)
plt.grid(True, alpha=0.3)




In [ ]:
optimal_k = 4
final_kmeans_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_cluster), # Uses the preprocessor defined in the previous cell
    ('cluster', KMeans(n_clusters=optimal_k, random_state=42, n_init=10))
])

# Fit to the data
final_kmeans_pipeline.fit(X_cluster)

#create a resutls dataframe
X_cluster['Cluster'] = final_kmeans_pipeline.named_steps['cluster'].labels_

In [ ]:
#Results from Kmeans
print("Size of each cluster")
print(X_cluster['Cluster'].value_counts().sort_index())
print("\n")

print("Cluster Average Numeric Variables")
print(X_cluster.groupby('Cluster')[num_features].mean())
print("\n")

print("Distribution of each position in each cluster")
print(X_cluster.groupby('Cluster')['Pos'].value_counts(normalize=True).unstack().fillna(0))


In [ ]:
# add data to graph
clusters = ['Durable Rotation', 'Elite Stars', 'Bench/Rookies', 'Fragile Veterans']
mpg = [28.3, 34.3, 14.1, 19.8]      # X-axis: Workload
injuries = [1.13, 2.45, 0.79, 1.95] # Y-axis: Risk

# Basic Scatter Plot
plt.scatter(mpg, injuries, color='blue')

# label each cluster
for i, label in enumerate(clusters):
    plt.text(mpg[i], injuries[i], label)

# Label
plt.title('Archetype Scatter: Workload versus Injury Risk')
plt.xlabel('Avg Minutes Per Game')
plt.ylabel('Avg Injuries Per Season')
plt.grid(True, alpha = 0.5)

plt.show()

Machine Learning
- Supervised Learning (Predict Games Missed based on variables) Ex. Injury Type, Body Part, Position
- 3 categories, low (1-2 games missed), medium (3-7 games missed), high (8 or more games missed)

In [ ]:
df = ml_final.copy()

# Create Target and Features
def severity_label(x):
    if x <= 2:
        return "Low"
    elif x <= 7:
        return "Medium"
    else:
        return "High"
    
df['Severity'] = df['GamesMissed'].apply(severity_label)


#Assign Features
features = ['InjuriesThisSeason', 'Age', 'Pos', 'MPG', "BodyPart", 'Team_y', 'SeasonPhase', "FG%", 'FT%', 'AVGPTS', 'AVGFTS', 'AVG3PTS', 'Injuries_Prev_Season', 'Rank_Prev_Season']
X = df[features]
y = df['Severity']

#Create a train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

#Pre Processing
cat_features = ['Pos', 'BodyPart', 'Team_y', 'SeasonPhase']
num_features = ['InjuriesThisSeason', 'Age', 'MPG', 'FG%', 'AVGPTS', 'AVGFTS', 'AVG3PTS', 'Injuries_Prev_Season', 'Rank_Prev_Season']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

In [ ]:
#Logistic Regression
log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42))
])


#Random Forest Model
#Preprocessing varibales so it encodes categoricals and keeps numerical 
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ))
])

In [ ]:
print("Logistic Regression Model Results")
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


print("Random Forest Model Results")
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))



In [ ]:
# Data
metrics = ['Accuracy', 'High Severity Precision']
log_scores = [0.61, 0.46]
rf_scores = [0.73, 0.86]

x = np.arange(len(metrics))
width = 0.35

# Basic Plot bar plot
plt.bar(x - width/2, log_scores, width, label='Logistic Regression')
plt.bar(x + width/2, rf_scores, width, label='Random Forest')

# Label axis and bars
plt.xticks(x, metrics)
plt.ylabel('Score')
plt.title('Model Comparison')
plt.legend()

plt.show()